In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [2]:
import torch
import torch.nn as nn

from src.preprocessing.dataset_loader import (
    build_dataset_index
)

from src.preprocessing.splitter import (
    create_stratified_split
)

from src.preprocessing.dataloaders import (
    create_datasets,
    create_dataloaders
)

from src.federated.client import (
    FederatedClient
)

from src.federated.server import (
    FederatedServer
)

from src.federated.trainer import (
    federated_training
)

In [3]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [4]:
train_paths, train_labels = build_dataset_index(
    "../data/raw/train"
)

test_paths, test_labels = build_dataset_index(
    "../data/raw/test"
)

In [5]:
(
    X_train,
    X_val,
    y_train,
    y_val
) = create_stratified_split(
    train_paths,
    train_labels
)

In [6]:
(
    train_dataset,
    val_dataset,
    test_dataset
) = create_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    test_paths,
    test_labels
)

In [7]:
(
    train_loader,
    val_loader,
    test_loader
) = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32
)

In [8]:
client1 = FederatedClient(
    client_id="client_1",
    train_loader=train_loader,
    device=device
)

client2 = FederatedClient(
    client_id="client_2",
    train_loader=train_loader,
    device=device
)

client3 = FederatedClient(
    client_id="client_3",
    train_loader=train_loader,
    device=device
)

clients = [
    client1,
    client2,
    client3
]

In [9]:
server = FederatedServer(
    device=device
)

In [10]:
criterion = nn.CrossEntropyLoss()

In [11]:
history = federated_training(

    server=server,

    clients=clients,

    val_loader=val_loader,

    criterion=criterion,

    device=device,

    rounds=5,

    local_epochs=1

)


========== Round 1/5 ==========
0.0002422919642413035 0.9994733929634094
0.002179633127525449 0.6736570000648499
0.00010379877494415268 0.9995606541633606
0.000571258831769228 0.6887548565864563
3.655797627288848e-05 0.99982088804245
0.0004617084050551057 0.6264695525169373
1.1228767107240856e-05 0.9999381303787231
0.00015321956016123295 0.636202335357666
7.032108987914398e-05 0.9998032450675964
0.0009389418410137296 0.6607021689414978
5.468963536259253e-06 0.9999643564224243
4.277304105926305e-05 0.6874701380729675
8.227122634707484e-06 0.999941349029541
0.00032483291579410434 0.7550222873687744
1.1535429393916274e-06 0.9999886751174927
2.5750276108738035e-05 0.6711165308952332
5.085233351564966e-05 0.9997382760047913
0.0010097032645717263 0.7807936668395996
6.050262527423911e-06 0.9999598264694214
0.0011560599086806178 0.7527428269386292
5.676082309946651e-06 0.9999567270278931
0.00047156642540358007 0.6429791450500488
8.516291927662678e-06 0.9999421834945679
0.0022520616184920073 0

RuntimeError: Parent directory ../models does not exist.

In [ ]:
print("\nTraining Finished\n")

print(history)